# Notebook 05 — Bonus: Database Sink (Option A, SQLite)

**Goal:** persist the nine aggregate tables in `data/warehouse.db` so Power BI can read from a database instead of loose CSV files.

How each requirement is met:

| Requirement | Implementation |
|---|---|
| Explicit DDL | `sql/schema.sql` — real types, primary keys, `NOT NULL` and `CHECK` constraints. `to_sql` is **not** used. |
| Idempotent loads | Truncate-and-reload: each table is emptied and refilled, so rerunning never duplicates rows (demonstrated below). |
| Transactions | Each table load runs inside `BEGIN … COMMIT`; any error triggers `ROLLBACK` (demonstrated below). |
| Indexes | On `year_month`, `mode` and `district` (defined in `schema.sql`). |
| Audit table | `pipeline_runs` (one row per run) + `pipeline_run_tables` (rows written per table per run). |
| Verification | Row counts and totals read back from the database are compared with the CSV files. |

In [12]:
import sqlite3
from datetime import datetime
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DB_PATH = PROJECT_ROOT / "data" / "warehouse.db"
SCHEMA_PATH = PROJECT_ROOT / "sql" / "schema.sql"

TABLES = [
    "agg_ridership_by_month", "agg_ridership_by_hour", "agg_route_performance",
    "agg_mode_summary", "agg_fare_type_mix", "agg_stop_activity",
    "agg_vehicle_utilisation", "agg_peak_analysis", "agg_transfer_behaviour",
]

## 1. Create the schema

`isolation_level=None` turns off Python's automatic transactions, so the notebook controls `BEGIN`, `COMMIT` and `ROLLBACK` itself.

In [13]:
def connect() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH, isolation_level=None)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


conn = connect()
conn.executescript(SCHEMA_PATH.read_text(encoding="utf-8"))

pd.read_sql_query(
    "SELECT type, name FROM sqlite_master WHERE name NOT LIKE 'sqlite_%' ORDER BY type DESC, name", conn
)

,type,name
0,table,agg_fare_type_mix
1,table,agg_mode_summary
2,table,agg_peak_analysis
3,table,agg_ridership_by_hour
4,table,agg_ridership_by_month
5,table,agg_route_performance
6,table,agg_stop_activity
7,table,agg_transfer_behaviour
8,table,agg_vehicle_utilisation
9,table,pipeline_run_tables


Eleven tables (nine aggregates plus two audit tables) and eight indexes exist. Because every statement uses `IF NOT EXISTS`, running the script again changes nothing.

## 2. Load functions

`load_table` inserts into the columns that the **database** defines, in that order. If a CSV is missing a column, or the data breaks a constraint (null key, duplicate primary key, wrong `CHECK` value), SQLite raises an error, the transaction is rolled back and the table keeps its previous contents.

In [14]:
def table_columns(conn: sqlite3.Connection, table: str) -> list[str]:
    return [row[1] for row in conn.execute(f"PRAGMA table_info({table})")]


def load_table(conn: sqlite3.Connection, table: str, df: pd.DataFrame) -> int:
    columns = table_columns(conn, table)
    missing = set(columns) - set(df.columns)
    if missing:
        raise ValueError(f"{table}: CSV is missing columns {sorted(missing)}")

    rows = df[columns].astype(object).where(df[columns].notna(), None).values.tolist()
    placeholders = ", ".join("?" for _ in columns)
    insert_sql = f"INSERT INTO {table} ({', '.join(columns)}) VALUES ({placeholders})"

    conn.execute("BEGIN")
    try:
        conn.execute(f"DELETE FROM {table}")          # truncate ...
        conn.executemany(insert_sql, rows)            # ... and reload
        conn.execute("COMMIT")
    except Exception:
        conn.execute("ROLLBACK")
        raise
    return len(rows)


def run_pipeline(conn: sqlite3.Connection) -> int:
    started = datetime.now().isoformat(timespec="seconds")
    run_id = conn.execute(
        "INSERT INTO pipeline_runs (started_at, status) VALUES (?, 'RUNNING')", (started,)
    ).lastrowid
    try:
        for table in TABLES:
            df = pd.read_csv(PROCESSED_DIR / f"{table}.csv")
            written = load_table(conn, table, df)
            conn.execute(
                "INSERT INTO pipeline_run_tables (run_id, table_name, rows_written) VALUES (?, ?, ?)",
                (run_id, table, written),
            )
        status, error = "SUCCESS", None
    except Exception as exc:
        status, error = "FAILED", f"{type(exc).__name__}: {exc}"
    conn.execute(
        "UPDATE pipeline_runs SET finished_at = ?, status = ?, error_message = ? WHERE run_id = ?",
        (datetime.now().isoformat(timespec="seconds"), status, error, run_id),
    )
    print(f"Run {run_id}: {status}" + (f" — {error}" if error else ""))
    return run_id

## 3. Run the load

In [15]:
run_id = run_pipeline(conn)
pd.read_sql_query(
    "SELECT table_name, rows_written FROM pipeline_run_tables WHERE run_id = ?", conn, params=(run_id,)
)

Run 5: SUCCESS


,table_name,rows_written
0,agg_fare_type_mix,30
1,agg_mode_summary,18
2,agg_peak_analysis,18
3,agg_ridership_by_hour,48
4,agg_ridership_by_month,6
5,agg_route_performance,360
6,agg_stop_activity,400
7,agg_transfer_behaviour,18
8,agg_vehicle_utilisation,180


## 4. Verify against the CSV files

For every table the row count and the sum of the main measure must be identical in the CSV and in the database.

In [16]:
checks = []
for table in TABLES:
    csv = pd.read_csv(PROCESSED_DIR / f"{table}.csv")
    db = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    measure = next(c for c in ["total_boardings", "boardings", "transfer_count"] if c in csv.columns)
    checks.append({
        "table": table,
        "csv_rows": len(csv),
        "db_rows": len(db),
        "measure": measure,
        "csv_sum": csv[measure].sum(),
        "db_sum": db[measure].sum(),
    })

checks = pd.DataFrame(checks)
checks["match"] = (checks["csv_rows"] == checks["db_rows"]) & (checks["csv_sum"] == checks["db_sum"])
assert checks["match"].all(), "Database and CSV files differ!"
checks

,table,csv_rows,db_rows,measure,csv_sum,db_sum,match
0,agg_ridership_by_month,6,6,total_boardings,85247,85247,True
1,agg_ridership_by_hour,48,48,total_boardings,85247,85247,True
2,agg_route_performance,360,360,total_boardings,85247,85247,True
3,agg_mode_summary,18,18,total_boardings,85247,85247,True
4,agg_fare_type_mix,30,30,boardings,85247,85247,True
5,agg_stop_activity,400,400,total_boardings,85247,85247,True
6,agg_vehicle_utilisation,180,180,total_boardings,85247,85247,True
7,agg_peak_analysis,18,18,total_boardings,85247,85247,True
8,agg_transfer_behaviour,18,18,transfer_count,30119,30119,True


All nine tables match the CSV files on both row count and measure total.

## 5. Idempotency: run it again

In [17]:
second_run = run_pipeline(conn)
after = {t: conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in TABLES}
assert after == dict(zip(checks["table"], checks["csv_rows"])), "Rerun changed the row counts!"
print("Row counts after the second run are unchanged:", after["agg_route_performance"], "route rows, etc.")

Run 6: SUCCESS
Row counts after the second run are unchanged: 360 route rows, etc.


The second run reloads the same data and the row counts stay exactly the same — no duplicates.

## 6. Transaction rollback

To prove the rollback works, a deliberately broken copy of the monthly table (one `year_month` set to null, which violates `NOT NULL`) is loaded. The load must fail **and** the table must still contain its six good rows afterwards.

In [18]:
broken = pd.read_csv(PROCESSED_DIR / "agg_ridership_by_month.csv")
broken.loc[0, "year_month"] = None

before = conn.execute("SELECT COUNT(*) FROM agg_ridership_by_month").fetchone()[0]
try:
    load_table(conn, "agg_ridership_by_month", broken)
except sqlite3.IntegrityError as exc:
    print("Load failed as expected:", exc)
after_rollback = conn.execute("SELECT COUNT(*) FROM agg_ridership_by_month").fetchone()[0]

assert before == after_rollback == 6
print(f"Rows before: {before}, rows after the failed load: {after_rollback} -> rollback worked")

Load failed as expected: NOT NULL constraint failed: agg_ridership_by_month.year_month
Rows before: 6, rows after the failed load: 6 -> rollback worked


## 7. Audit trail

In [19]:
pd.read_sql_query(
    """
    SELECT r.run_id, r.started_at, r.finished_at, r.status,
           COUNT(t.table_name) AS tables_loaded, COALESCE(SUM(t.rows_written), 0) AS rows_written
    FROM pipeline_runs r
    LEFT JOIN pipeline_run_tables t ON t.run_id = r.run_id
    GROUP BY r.run_id
    ORDER BY r.run_id DESC
    LIMIT 5
    """,
    conn,
)

,run_id,started_at,finished_at,status,tables_loaded,rows_written
0,6,2026-10-28T17:05:47,2026-10-28T17:05:47,SUCCESS,9,1078
1,5,2026-10-28T17:05:47,2026-10-28T17:05:47,SUCCESS,9,1078
2,4,2026-10-28T17:05:44,2026-10-28T17:05:44,SUCCESS,9,1078
3,3,2026-10-28T17:05:44,2026-10-28T17:05:44,SUCCESS,9,1078
4,2,2026-10-26T17:07:32,2026-10-26T17:07:32,SUCCESS,9,1078


Each run is recorded with its start and end time, status and the number of rows written to each table.

## 8. Example query that Power BI could use

In [20]:
pd.read_sql_query(
    """
    SELECT m.year_month, m.mode, m.total_boardings, m.boardings_per_route,
           ROUND(m.total_boardings * 1.0 / t.total_boardings, 4) AS share_of_network,
           tr.transfer_rate
    FROM agg_mode_summary m
    JOIN agg_ridership_by_month t  ON t.year_month = m.year_month
    JOIN agg_transfer_behaviour tr ON tr.year_month = m.year_month AND tr.mode = m.mode
    WHERE m.year_month = '2024-06'
    ORDER BY m.total_boardings DESC
    """,
    conn,
)

,year_month,mode,total_boardings,boardings_per_route,share_of_network,transfer_rate
0,2024-06,Bus,7729,227.32,0.5551,0.3897
1,2024-06,Tram,4788,239.40,0.3439,0.3763
2,2024-06,Metro,1407,234.50,0.1010,0.3753


In [21]:
conn.close()

## 9. Connecting Power BI (manual step)

1. Install the SQLite ODBC driver (for example *SQLite3 ODBC Driver* by Christian Werner) and create a DSN that points to `data/warehouse.db`.
2. In Power BI Desktop: **Get Data → ODBC →** choose the DSN → select the `agg_*` tables.
3. Rebuild one page (for example *Network Overview*) from these database tables instead of the CSV files.

Power BI Desktop runs only on Windows, so this step is done in the report itself and is not part of the notebook.